# Baseline Notebook — Customer Churn Retention TriggerPurpose: establish a **non-ML baseline** and a **minimal ML baseline** on the same data, to test whether ML is justified before building anything more complex.See the accompanying ML Problem-Framing Memo, Responsible Data Card, and Risk Register for context.

In [ ]:
import pandas as pdimport numpy as npdf = pd.read_csv('customer-churn-training.csv')df

## 1. Class balanceSmall sample (n=12). Any accuracy number here should be read with heavy caveats.

In [ ]:
print("Rows:", len(df))print(df['churned'].value_counts())print("Churn rate:", df['churned'].mean())

## 2. Non-ML baseline: a 2-line rule`last_login_days > 10 OR support_tickets >= 3`This encodes the same intuition a support/CS lead would use without any model: customers who've gone quiet or are filing lots of tickets are at risk.

In [ ]:
df['rule_pred'] = ((df['last_login_days'] > 10) | (df['support_tickets'] >= 3)).astype(int)def confusion(y_true, y_pred):    tp = ((y_true==1)&(y_pred==1)).sum()    tn = ((y_true==0)&(y_pred==0)).sum()    fp = ((y_true==0)&(y_pred==1)).sum()    fn = ((y_true==1)&(y_pred==0)).sum()    prec = tp/(tp+fp) if (tp+fp) else float('nan')    rec  = tp/(tp+fn) if (tp+fn) else float('nan')    acc  = (tp+tn)/len(y_true)    return dict(tp=tp, fp=fp, tn=tn, fn=fn, accuracy=acc, precision=prec, recall=rec)confusion(df['churned'], df['rule_pred'])

{'tp': 5, 'fp': 0, 'tn': 7, 'fn': 0, 'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0}

**Result: the rule alone gets every case right on this sample (5/5 churners caught, 0 false alarms).**

## 3. Minimal ML baseline: Logistic Regression (Leave-One-Out CV)With only 12 rows, a train/test split isn't meaningful, so we use leave-one-out cross-validation as the most honest evaluation this data size allows.

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import LeaveOneOutfrom sklearn.preprocessing import StandardScalerX = df[['tenure_months','support_tickets','monthly_spend_inr','last_login_days']].copy()X = pd.concat([X, pd.get_dummies(df['plan_type'], prefix='plan')], axis=1)y = df['churned'].valuesloo = LeaveOneOut()preds = []for train_idx, test_idx in loo.split(X):    scaler = StandardScaler()    X_train = scaler.fit_transform(X.iloc[train_idx])    X_test = scaler.transform(X.iloc[test_idx])    clf = LogisticRegression(max_iter=1000).fit(X_train, y[train_idx])    preds.append(clf.predict(X_test)[0])confusion(y, np.array(preds))

{'tp': 5, 'fp': 0, 'tn': 7, 'fn': 0, 'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0}

## 4. Baseline comparison and conclusion| Model | Accuracy | Precision | Recall ||---|---|---|---|| Majority class ("predict no churn") | 0.58 | n/a | 0.00 || **Non-ML rule** (login>10 OR tickets>=3) | **1.00** | **1.00** | **1.00** || Logistic Regression (LOOCV) | 1.00 | 1.00 | 1.00 |**The 2-line rule matches the ML model exactly.** On the data available today, ML adds engineering and monitoring cost without adding predictive power. See the memo's *ML-justification verdict* for the recommendation and the conditions under which that verdict should be revisited.**Caution:** a perfectly separable 12-row sample is itself a data-quality signal, not just a modeling result — see the Responsible Data Card, leakage-risk section, before trusting either number above.